# Sanitize & Extract — PII Redaction + Structured Fields

Stage 2 of the real-offer pipeline. Input: the normalized texts from
Stage 1 (`data/raw/`). Output: PII-redacted texts (`data/redacted/text/`),
cached LLM block detections (`data/redacted/pii_blocks/`) and structured
fields (`data/extracted/`).

**Architecture (LLM-first, one masking pass):**

1. **LLM block detection** runs *first* on the **unmasked** text head —
   it sees the full letterhead and reports sender + customer strings
   verbatim. Results are cached per offer (resumable, reviewable).
2. **⏸️ Stop point** — review the detections before anything is masked.
3. **One combined masking pass** per offer:
   - (a) regex sanitizer (`PIISanitizer` from `01-submission/`) —
     IBAN / BIC / email / phone / USt-IdNr / Steuernummer, deterministic;
   - (b) dynamic replacement of the LLM-reported strings (containment
     check: only strings that literally occur are replaced).
   If the LLM ever reports a string that *contains* a phone number or
   email, the regex pass masks it first, the containment check fails,
   nothing is replaced — and the re-scan catches the remainder. Safe
   failure mode.
4. **Extraction** — `preis` per LLM (explicit total only), `datum`
   reused from Stage 1 (deterministic).
5. **PII re-scan (hard gate)** — regex detect + known real patterns +
   every LLM-reported string must be gone. Zero hits or abort.

Redaction policy:

| Block | Treatment |
|---|---|
| Sender street | `[ADRESSE_REDACTED]` |
| Sender PLZ/Ort | `[ADRESSE_REDACTED]` (only on sender lines / footer) |
| Sender name + website | **kept** (the company is the product) |
| Customer firma / ansprechpartner | `[KUNDE_REDACTED]` |
| Customer street | `[ADRESSE_REDACTED]` |
| Customer PLZ/Ort | **kept** (distance inference is a feature) |

In [ ]:
import os, re, json, time
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Load env files (walk up: notebooks/ -> repo root). .env.example provides
# the defaults, .env overrides them (same layering as the app's config.py).
for path in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    example, env = path / ".env.example", path / ".env"
    if example.exists():
        load_dotenv(example, override=False)
        print(f"✅ Loaded defaults from {example}")
    if env.exists():
        load_dotenv(env, override=True)
        print(f"✅ Loaded .env from {env}")
    if example.exists() or env.exists():
        break

# Secrets layer (outside the repo, chmod 600): highest priority, keeps API
# keys out of the workspace (same as the app's config.py).
_secrets = Path.home() / ".config" / "rag-quote-history" / "secrets.env"
if _secrets.exists():
    load_dotenv(_secrets, override=True)
    print(f"\u2705 Loaded secrets from {_secrets}")

LLM_BASE_URL = os.getenv("LLM_BASE_URL", "")
LLM_MODEL = os.getenv("LLM_MODEL", "")
LLM_API_KEY = os.getenv("LLM_API_KEY", "")

def redact_url(url: str) -> str:
    """Mask IP address and port in a URL for display."""
    url = re.sub(r"\d{1,3}(?:\.\d{1,3}){3}", "*.*.*.*", url)
    return re.sub(r":\d+", ":****", url)

DEMO_DIR = Path.cwd().parent            # 01-submission
RAW_DIR = DEMO_DIR / "data" / "raw"
REDACTED_TEXT_DIR = DEMO_DIR / "data" / "redacted" / "text"
PII_BLOCKS_DIR = DEMO_DIR / "data" / "redacted" / "pii_blocks"
EXTRACTED_DIR = DEMO_DIR / "data" / "extracted"
for d in (REDACTED_TEXT_DIR, PII_BLOCKS_DIR, EXTRACTED_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Reuse the sanitizer from this package (single source of truth — do not fork)
import sys
sys.path.insert(0, str(DEMO_DIR))
import importlib, sanitizer as _san_mod
importlib.reload(_san_mod)
PIISanitizer = _san_mod.PIISanitizer

client = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)

n_raw = len([f for f in RAW_DIR.glob("AG*.txt") if not f.name.endswith(".ref.txt")])
print(f"📁 Raw texts:   {RAW_DIR} ({n_raw})")
print(f"📁 Redacted:    {REDACTED_TEXT_DIR}")
print(f"📁 PII blocks:  {PII_BLOCKS_DIR}")
print(f"📁 Extracted:   {EXTRACTED_DIR}")
print(f"🤖 LLM: {LLM_MODEL} at {redact_url(LLM_BASE_URL)}")


In [ ]:
# Set CLEAN_SLATE = True to rebuild redacted/ and extracted/ from scratch.
# Default False: finished offers are skipped via the per-offer caches.
import shutil

CLEAN_SLATE = False

if CLEAN_SLATE:
    for d in (REDACTED_TEXT_DIR, PII_BLOCKS_DIR, EXTRACTED_DIR):
        if d.exists():
            shutil.rmtree(d)
            print(f"Removed {d}")
        d.mkdir(parents=True, exist_ok=True)
    print("Clean slate — redacted texts, PII blocks and extractions will be rebuilt")
else:
    print("Resume mode — existing caches are kept")


## Step 0: Load Raw Texts

The normalized texts from Stage 1. These still contain real PII —
they are the *input* of this notebook and never leave `data/raw/`.

In [ ]:
raw_texts = {}
for f in sorted(RAW_DIR.glob("AG*.txt")):
    if f.name.endswith(".ref.txt"):      # skip pdfplumber reference files
        continue
    raw_texts[f.stem] = f.read_text()
print(f"Loaded {len(raw_texts)} raw texts")
assert len(raw_texts) > 0, f"no raw texts found in {RAW_DIR}"


## Step 1: LLM Block Detection (Sender + Customer)

The LLM reads the **unmasked** text head (letterhead + addressees) and
reports the sender and customer blocks **verbatim**. One call per offer,
cached in `data/redacted/pii_blocks/AG####.json` — resumable and
manually reviewable.

Nothing is masked in this step. The detections are only *input* for the
combined masking pass in Step 2.

In [ ]:
BLOCK_PROMPT = """Dies ist der Kopf eines deutschen Projektangebots (normalisierter Text).
Identifiziere zwei Blöcke:

1. ABSENDER (der Anbieter, von dem das Angebot kommt):
   - strassen: Straßenadressen des Absenders (Straße + Hausnummer), wörtlich
   - ort: PLZ + Ort des Absenders, wörtlich (z.B. "24106 Kiel")

2. KUNDE (der Auftraggeber, an den das Angebot gerichtet ist):
   - firma: Firmenname des Kunden, wörtlich
   - ansprechpartner: Name der Ansprechperson, wörtlich (null wenn keiner)
   - strassen: Straßenadressen des Kunden (Straße + Hausnummer), wörtlich
   - ort: PLZ + Ort des Kunden, wörtlich (null wenn keiner)

Regeln:
- Nur Strings, die WÖRTLICH im Text stehen. Keine Ergänzungen, keine Korrekturen.
- Der Absender ist NICHT der Kunde.
- Antworte mit EXAKT einem JSON-Objekt, keine Markdown-Fences:
{{"sender": {{"strassen": [], "ort": null}}, "kunde": {{"firma": null, "ansprechpartner": null, "strassen": [], "ort": null}}}}

TEXT:
{text}"""

HEAD_LINES = 60   # the letterhead + addressees always sit in the first lines

def _parse_json(raw: str) -> dict:
    raw = re.sub(r"```(?:json)?\s*|\s*```", "", raw).strip()
    start, end = raw.find("{"), raw.rfind("}")
    if start == -1 or end <= start:
        raise json.JSONDecodeError("Kein JSON in Antwort", raw[:200], 0)
    data = json.loads(raw[start:end + 1])
    if not isinstance(data, dict):
        raise json.JSONDecodeError("Antwort ist kein Objekt", raw[:200], 0)
    return data

def detect_blocks(offer_id: str, text: str, retries: int = 3) -> dict:
    """LLM call: sender + customer blocks from the unmasked text head."""
    head = "\n".join(text.splitlines()[:HEAD_LINES])
    prompt = BLOCK_PROMPT.format(text=head)
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            resp = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=800,
                extra_body={"chat_template_kwargs": {"enable_thinking": False}},
            )
            return _parse_json(resp.choices[0].message.content)
        except (json.JSONDecodeError, ValueError) as e:
            last_err = e
            print(f"(attempt {attempt} failed: {e})", end=" ")
            if attempt < retries:
                time.sleep(2 * attempt)
    raise last_err

# --- detect (cached, resumable) ---
blocks = {}
for p in sorted(PII_BLOCKS_DIR.glob("*.json")):
    try:
        blocks[p.stem] = json.loads(p.read_text())
    except json.JSONDecodeError:
        print(f"Skip corrupt file {p.name}")
if blocks:
    print(f"Reloaded {len(blocks)} cached PII blocks")

todo = [ag for ag in sorted(raw_texts) if ag not in blocks]
print(f"Detecting blocks for {len(todo)} offers ...\n")
t0 = time.time()
for i, ag in enumerate(todo, 1):
    print(f"[{i}/{len(todo)}] {ag} ...", end=" ", flush=True)
    try:
        b = detect_blocks(ag, raw_texts[ag])
        blocks[ag] = b
        (PII_BLOCKS_DIR / f"{ag}.json").write_text(json.dumps(b, ensure_ascii=False, indent=2))
        k = b.get("kunde") or {}
        print(f"OK  sender={len(b.get('sender', {}).get('strassen', []))} str  "
              f"kunde={k.get('firma') or k.get('ansprechpartner') or 'n/a'}")
    except Exception as e:
        print(f"FAIL {type(e).__name__}: {e}")
print(f"\n{len(blocks)}/{len(raw_texts)} blocks in {PII_BLOCKS_DIR.name}/ ({time.time()-t0:.0f}s this run)")


## ⏸️ Stop Point — Review the Detections

**Do not run Step 2 yet.** Open `data/redacted/pii_blocks/AG####.json`
and spot-check the detections:

- Is the **sender** right? (street + PLZ/Ort of the company)
- Is the **customer** right? (firma / ansprechpartner / street / PLZ/Ort)
- Any string that is *not* verbatim in the text? (it will simply not be
  replaced — the containment check guards against that)
- Any PII the LLM **missed**? (the re-scan in Step 4 is the backstop)

To correct a detection: edit the JSON file (or delete it for a re-run)
and re-run Step 1. Then continue with Step 2.

## Step 2: Combined Masking (Regex → Dynamic)

One pass per offer, two layers:

1. **Regex** — `PIISanitizer.mask()` (IBAN / BIC / email / phone /
   USt-IdNr / Steuernummer). Deterministic, runs first.
2. **Dynamic** — replacement of the LLM-reported strings, with a
   containment check (only strings that occur are replaced). Two
   matching layers per string:
   - *exact-flex*: case- and spacing-insensitive (catches `Ostelabs`
     vs `osteolabs`, glued PDF artifacts like `Gutenbergstraße18`);
   - *fuzzy*: near-variants of name-like runs (similarity ≥ 0.85,
     catches spelling variants such as `Ostelabs GmbH` vs the reported
     `osteolabs GmbH`).
   Treatment:
   - sender street → `[ADRESSE_REDACTED]` (global)
   - sender PLZ/Ort → `[ADRESSE_REDACTED]` on lines carrying a sender
     street, the line right below one, or OFFICE/STUDIO footer lines —
     so a customer in the same city keeps their PLZ/Ort
   - customer firma / ansprechpartner → `[KUNDE_REDACTED]` (global)
   - customer street → `[ADRESSE_REDACTED]` (customer PLZ/Ort **stays**)

The sender's name and website are intentionally kept — the company is
the product. Results are written to `data/redacted/text/`.

In [ ]:
sanitizer = PIISanitizer()

import difflib

def _flex(s: str) -> re.Pattern:
    """Case-insensitive pattern with flexible spacing between tokens.

    Catches spelling variants (Ostelabs vs osteolabs) and glued PDF
    artifacts (Gutenbergstraße18 vs Gutenbergstraße 18).
    """
    return re.compile(r"[ \t]*".join(re.escape(t) for t in s.split()), re.IGNORECASE)

# Name-like runs (words with letters/digits, no digits-only, no pure
# punctuation) — candidates for fuzzy variant matching.
NAME_RUN = re.compile(r"[A-Za-zÄÖÜäöüß][A-Za-zÄÖÜäöüß0-9&+\-.]*(?:[ \t][A-Za-zÄÖÜäöüß0-9&+\-.]*)*")
FUZZY_THRESHOLD = 0.85

def _ratio(a: str, b: str) -> float:
    a = re.sub(r"\s+", " ", a.lower()).strip()
    b = re.sub(r"\s+", " ", b.lower()).strip()
    if a == b:
        return 1.0
    return difflib.SequenceMatcher(None, a, b).ratio()

def _fuzzy_replace(text: str, s: str, marker: str):
    """Replace name runs that are near-variants of s (e.g. 'Ostelabs GmbH'
    vs reported 'osteolabs GmbH'). Returns (text, n_replaced)."""
    if not s or len(s) < 4:
        return text, 0
    n = 0
    def sub(m):
        nonlocal n
        cand = m.group(0)
        if len(cand) >= 4 and _ratio(s, cand) >= FUZZY_THRESHOLD:
            n += 1
            return marker
        return cand
    return NAME_RUN.sub(sub, text), n

def _fuzzy_present(text: str, s: str) -> bool:
    if not s or len(s) < 4:
        return False
    return any(_ratio(s, m.group(0)) >= FUZZY_THRESHOLD
               for m in NAME_RUN.finditer(text) if len(m.group(0)) >= 4)

def _street_in_line(ln: str, streets: list) -> bool:
    return any(_flex(st).search(ln) or _fuzzy_present(ln, st) for st in streets)

def apply_block_redaction(text: str, blocks: dict, raw_text: str = None):
    """Deterministic replacement of LLM-reported blocks.

    Returns (masked_text, replaced). Two matching layers per string:
    exact-flex (case/spacing-insensitive) first, then fuzzy near-variant
    matching (catches spelling variants like 'Ostelabs' vs 'osteolabs').

    raw_text: the unmasked text — sender street lines are detected on it,
    because the regex layer has already replaced streets with
    [ADRESSE_REDACTED] in the masked text (line counts are identical).
    """
    replaced = []
    sender = blocks.get("sender") or {}
    kunde = blocks.get("kunde") or {}
    s_strassen = [s.strip() for s in (sender.get("strassen") or []) if s and len(s.strip()) >= 4]
    s_ort = (sender.get("ort") or "").strip()

    # 1) sender PLZ/Ort — on lines carrying a sender street, the line right
    #    below such a line, or OFFICE/STUDIO footer lines. A customer in the
    #    same city keeps their PLZ/Ort (in this corpus a customer PLZ line is
    #    never adjacent to a sender street line).
    if s_ort:
        pat_ort = _flex(s_ort)
        lines = text.splitlines()
        ref_lines = (raw_text or text).splitlines()
        street_idx = {i for i, ln in enumerate(ref_lines) if _street_in_line(ln, s_strassen)}
        for i, ln in enumerate(lines):
            if pat_ort.search(ln) and (i in street_idx or i - 1 in street_idx
                                       or re.search(r"\b(OFFICE|STUDIO)\b", ln)):
                lines[i] = pat_ort.sub("[ADRESSE_REDACTED]", ln)
                replaced.append(("sender.ort", s_ort))
        text = "\n".join(lines)
    # 2) sender streets — global (exact-flex, then fuzzy)
    for st in s_strassen:
        if _flex(st).search(text):
            text = _flex(st).sub("[ADRESSE_REDACTED]", text)
            replaced.append(("sender.strasse", st))
        text, n = _fuzzy_replace(text, st, "[ADRESSE_REDACTED]")
        if n:
            replaced.append(("sender.strasse~", st))
    # 3) customer company / contact person — global
    for key in ("firma", "ansprechpartner"):
        v = (kunde.get(key) or "").strip()
        if v and len(v) >= 4:
            if _flex(v).search(text):
                text = _flex(v).sub("[KUNDE_REDACTED]", text)
                replaced.append((f"kunde.{key}", v))
            text, n = _fuzzy_replace(text, v, "[KUNDE_REDACTED]")
            if n:
                replaced.append((f"kunde.{key}~", v))
    # 4) customer streets — global (PLZ/Ort intentionally kept)
    for st in [s.strip() for s in (kunde.get("strassen") or []) if s and len(s.strip()) >= 4]:
        if _flex(st).search(text):
            text = _flex(st).sub("[ADRESSE_REDACTED]", text)
            replaced.append(("kunde.strasse", st))
        text, n = _fuzzy_replace(text, st, "[ADRESSE_REDACTED]")
        if n:
            replaced.append(("kunde.strasse~", st))
    return text, replaced

# --- one combined pass: regex first, then dynamic ---
final_texts = {}
total_replaced = 0
regex_totals = {}
for ag, raw in sorted(raw_texts.items()):
    report = sanitizer.mask(raw)                      # (a) regex layer
    for k, v in report.found.items():
        regex_totals[k] = regex_totals.get(k, 0) + v
    text = report.masked_text
    b = blocks.get(ag)
    if b:
        text, replaced = apply_block_redaction(text, b, raw)   # (b) dynamic layer
        total_replaced += len(replaced)
    final_texts[ag] = text
    (REDACTED_TEXT_DIR / f"{ag}.txt").write_text(text)

print("Regex layer totals:", regex_totals)
print(f"Dynamic layer: replaced {total_replaced} block strings")
print(f"Wrote {len(final_texts)} redacted texts to {REDACTED_TEXT_DIR}")


## Step 3: Structured Extraction (`preis`)

`preis` is extracted per offer by the LLM — **explicit total only**
(no summing, no guessing; `null` when the document has no explicit
Gesamtpreis). `datum` is reused from the Stage 1 JSON (deterministic,
from the pdfplumber reference text). Cached in
`data/extracted/AG####.json`.

In [ ]:
EXTRACTION_PROMPT = """Extrahiere aus dem folgenden Post-Production-Projektangebot den Gesamtpreis.

WICHTIG — Ausgabeformat:
Deine Antwort muss EXAKT EIN flaches JSON-Objekt mit genau diesem Key enthalten (keine Markdown-Fences, kein Text nach dem JSON):
{{"preis": null}}

Bedeutung:
- "preis": Gesamtpreis netto in EUR als Zahl (ohne Tausenderpunkte).
  NUR wenn der Gesamtpreis EXPLIZIT im Dokument ausgeschrieben ist
  (z.B. "Gesamt", "Gesamtpreis", "Summe"). Schätze, berechne oder summiere NICHT
  selbst — wenn kein expliziter Gesamtpreis steht, setze null.

ANGEBOT:
{text}"""

def extract_preis(offer_id: str, text: str, retries: int = 3) -> dict:
    prompt = EXTRACTION_PROMPT.format(text=text)
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            resp = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=200,
                extra_body={"chat_template_kwargs": {"enable_thinking": False}},
            )
            raw = re.sub(r"```(?:json)?\s*|\s*```", "", resp.choices[0].message.content).strip()
            start, end = raw.find("{"), raw.rfind("}")
            data = json.loads(raw[start:end + 1])
            if "preis" not in data:
                raise ValueError("Missing key: preis")
            return {"angebot_id": offer_id, "preis": data["preis"]}
        except (json.JSONDecodeError, ValueError) as e:
            last_err = e
            print(f"(attempt {attempt} failed: {e})", end=" ")
            if attempt < retries:
                time.sleep(2 * attempt)
    raise last_err

extracted = {}
for p in sorted(EXTRACTED_DIR.glob("*.json")):
    try:
        extracted[p.stem] = json.loads(p.read_text())
    except json.JSONDecodeError:
        print(f"Skip corrupt file {p.name}")
if extracted:
    print(f"Reloaded {len(extracted)} existing extractions")

todo = [ag for ag in sorted(final_texts) if ag not in extracted]
print(f"Extracting preis for {len(todo)} offers ...\n")
t0 = time.time()
for i, ag in enumerate(todo, 1):
    print(f"[{i}/{len(todo)}] {ag} ...", end=" ", flush=True)
    try:
        d = extract_preis(ag, final_texts[ag])
        # reuse datum from Stage 1 (deterministic, from the pdfplumber reference)
        stage1 = json.loads((RAW_DIR / f"{ag}.json").read_text())
        d["datum"] = stage1.get("datum")
        extracted[ag] = d
        (EXTRACTED_DIR / f"{ag}.json").write_text(json.dumps(d, ensure_ascii=False, indent=2))
        print(f"OK  preis={d['preis']}  datum={d['datum']}")
    except Exception as e:
        print(f"FAIL {type(e).__name__}: {e}")
print(f"\n{len(extracted)}/{len(final_texts)} extractions in {EXTRACTED_DIR.name}/ ({time.time()-t0:.0f}s this run)")


## Step 4: PII Re-Scan (Hard Gate)

Three independent checks on every redacted text — **zero hits or abort**:

1. **Regex detect** — `PIISanitizer.detect()` (IBAN / BIC / email /
   phone / USt-IdNr / Steuernummer).
2. **Known real patterns** — the concrete values from `AGENTS.md`
   (USt-IdNr, Steuernummer, customer name, IBAN prefixes, BICs, phone).
3. **Every LLM-reported string** — sender streets, sender PLZ/Ort
   (on sender-context lines), customer firma / ansprechpartner /
   streets must all be gone, checked with the same exact-flex + fuzzy
   matching as the masking pass.

If this passes, `data/redacted/` and `data/extracted/` are safe to
expose (index, app, zip, presentation).

In [ ]:
import difflib

def _flex(s: str) -> re.Pattern:
    """Case-insensitive pattern with flexible spacing between tokens.

    Catches spelling variants (Ostelabs vs osteolabs) and glued PDF
    artifacts (Gutenbergstraße18 vs Gutenbergstraße 18).
    """
    return re.compile(r"[ \t]*".join(re.escape(t) for t in s.split()), re.IGNORECASE)

# Name-like runs (words with letters/digits, no digits-only, no pure
# punctuation) — candidates for fuzzy variant matching.
NAME_RUN = re.compile(r"[A-Za-zÄÖÜäöüß][A-Za-zÄÖÜäöüß0-9&+\-.]*(?:[ \t][A-Za-zÄÖÜäöüß0-9&+\-.]*)*")
FUZZY_THRESHOLD = 0.85

def _ratio(a: str, b: str) -> float:
    a = re.sub(r"\s+", " ", a.lower()).strip()
    b = re.sub(r"\s+", " ", b.lower()).strip()
    if a == b:
        return 1.0
    return difflib.SequenceMatcher(None, a, b).ratio()

def _fuzzy_replace(text: str, s: str, marker: str):
    """Replace name runs that are near-variants of s (e.g. 'Ostelabs GmbH'
    vs reported 'osteolabs GmbH'). Returns (text, n_replaced)."""
    if not s or len(s) < 4:
        return text, 0
    n = 0
    def sub(m):
        nonlocal n
        cand = m.group(0)
        if len(cand) >= 4 and _ratio(s, cand) >= FUZZY_THRESHOLD:
            n += 1
            return marker
        return cand
    return NAME_RUN.sub(sub, text), n

def _fuzzy_present(text: str, s: str) -> bool:
    if not s or len(s) < 4:
        return False
    return any(_ratio(s, m.group(0)) >= FUZZY_THRESHOLD
               for m in NAME_RUN.finditer(text) if len(m.group(0)) >= 4)

def _street_in_line(ln: str, streets: list) -> bool:
    return any(_flex(st).search(ln) or _fuzzy_present(ln, st) for st in streets)

def apply_block_redaction(text: str, blocks: dict):
    """Deterministic replacement of LLM-reported blocks.

    Returns (masked_text, replaced). Two matching layers per string:
    exact-flex (case/spacing-insensitive) first, then fuzzy near-variant
    matching (catches spelling variants like 'Ostelabs' vs 'osteolabs').
    """
    replaced = []
    sender = blocks.get("sender") or {}
    kunde = blocks.get("kunde") or {}
    s_strassen = [s.strip() for s in (sender.get("strassen") or []) if s and len(s.strip()) >= 4]
    s_ort = (sender.get("ort") or "").strip()

    # 1) sender PLZ/Ort — on lines carrying a sender street, the line right
    #    below such a line, or OFFICE/STUDIO footer lines. A customer in the
    #    same city keeps their PLZ/Ort (in this corpus a customer PLZ line is
    #    never adjacent to a sender street line).
    if s_ort:
        pat_ort = _flex(s_ort)
        lines = text.splitlines()
        street_idx = {i for i, ln in enumerate(lines) if _street_in_line(ln, s_strassen)}
        for i, ln in enumerate(lines):
            if pat_ort.search(ln) and (i in street_idx or i - 1 in street_idx
                                       or re.search(r"\b(OFFICE|STUDIO)\b", ln)):
                lines[i] = pat_ort.sub("[ADRESSE_REDACTED]", ln)
                replaced.append(("sender.ort", s_ort))
        text = "\n".join(lines)
    # 2) sender streets — global (exact-flex, then fuzzy)
    for st in s_strassen:
        if _flex(st).search(text):
            text = _flex(st).sub("[ADRESSE_REDACTED]", text)
            replaced.append(("sender.strasse", st))
        text, n = _fuzzy_replace(text, st, "[ADRESSE_REDACTED]")
        if n:
            replaced.append(("sender.strasse~", st))
    # 3) customer company / contact person — global
    for key in ("firma", "ansprechpartner"):
        v = (kunde.get(key) or "").strip()
        if v and len(v) >= 4:
            if _flex(v).search(text):
                text = _flex(v).sub("[KUNDE_REDACTED]", text)
                replaced.append((f"kunde.{key}", v))
            text, n = _fuzzy_replace(text, v, "[KUNDE_REDACTED]")
            if n:
                replaced.append((f"kunde.{key}~", v))
    # 4) customer streets — global (PLZ/Ort intentionally kept)
    for st in [s.strip() for s in (kunde.get("strassen") or []) if s and len(s.strip()) >= 4]:
        if _flex(st).search(text):
            text = _flex(st).sub("[ADRESSE_REDACTED]", text)
            replaced.append(("kunde.strasse", st))
        text, n = _fuzzy_replace(text, st, "[ADRESSE_REDACTED]")
        if n:
            replaced.append(("kunde.strasse~", st))
    return text, replaced

KNOWN_REAL_PATTERNS = [
    "DE 318 492 915", "DE318492915",   # USt-IdNr (both spellings)
    "20/048/61147",                     # Steuernummer
    "Jarre & Kahl",                     # customer name
    "DE11 2004 1133", "DE89 3704 0044", # IBAN prefixes
    "COBADEHD001", "COBADEFFXXX",       # BICs
    "0151 54 154 032", "015154154032",  # phone (both spellings)
]

violations = {}
for ag, text in sorted(final_texts.items()):
    hits = []
    # 1) regex-detectable PII
    found = sanitizer.detect(text)
    if found:
        hits.append(f"regex: {found}")
    # 2) known real patterns
    for pat in KNOWN_REAL_PATTERNS:
        if pat in text:
            hits.append(f"known: {pat!r}")
    # 3) every LLM-reported string must be gone (exact-flex + fuzzy)
    b = blocks.get(ag) or {}
    s_strassen = [s.strip() for s in (b.get("sender", {}).get("strassen") or []) if s.strip()]
    s_ort = (b.get("sender", {}).get("ort") or "").strip()
    for v in s_strassen:
        if _flex(v).search(text) or _fuzzy_present(text, v):
            hits.append(f"reported: {v!r}")
    if s_ort:
        lines = text.splitlines()
        raw_lines = raw_texts[ag].splitlines()   # streets are gone in the masked text
        k_strassen = [x.strip() for x in (b.get("kunde", {}).get("strassen") or []) if x.strip()]
        for i, ln in enumerate(lines):
            if not _flex(s_ort).search(ln):
                continue
            # (a) sender street on this line / the line above (raw text)
            sender_ctx = (_street_in_line(raw_lines[i], s_strassen)
                          or (i > 0 and _street_in_line(raw_lines[i - 1], s_strassen))
                          or re.search(r"\b(OFFICE|STUDIO)\b", ln))
            # (b) redaction tag on this line — but only if the line is NOT a
            #     customer street line (customer PLZ/Ort stays by policy)
            tag_ctx = ("[ADRESSE_REDACTED]" in ln
                       and not _street_in_line(raw_lines[i], k_strassen)
                       and not (i > 0 and _street_in_line(raw_lines[i - 1], k_strassen)))
            if sender_ctx or tag_ctx:
                hits.append(f"reported: {s_ort!r}")
                break
    for key in ("firma", "ansprechpartner"):
        v = (b.get("kunde", {}).get(key) or "").strip()
        if v and (_flex(v).search(text) or _fuzzy_present(text, v)):
            hits.append(f"reported: {v!r}")
    for v in [s.strip() for s in (b.get("kunde", {}).get("strassen") or []) if s.strip()]:
        if _flex(v).search(text) or _fuzzy_present(text, v):
            hits.append(f"reported: {v!r}")
    if hits:
        violations[ag] = hits

if violations:
    for ag, hits in violations.items():
        print(f"❌ {ag}: {hits}")
    raise AssertionError(f"PII re-scan failed: {len(violations)} offers with violations")

print(f"✅ PII re-scan passed: {len(final_texts)} redacted texts, zero hits")
print("   data/redacted/ and data/extracted/ are safe to expose.")


## Done

Stage 2 outputs:

- `data/redacted/text/AG####.txt` — PII-redacted texts (safe to expose)
- `data/redacted/pii_blocks/AG####.json` — LLM block detections (cache)
- `data/extracted/AG####.json` — `angebot_id`, `preis`, `datum`

**Next:** Stage 3 (`03-build-index.ipynb`) — chunk the redacted texts,
embed with `nomic-embed-text`, persist to `data/db/chroma/`
(collection `offers`) + `data/db/sql/`.